In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
df

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [4]:
df.drop(['flags','intent'],axis = 1,inplace = True)

In [5]:
df

,instruction,category,response
0,question about cancelling order {{Order Number}},ORDER,I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,ORDER,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},ORDER,I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},ORDER,I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",ORDER,I'm sensitive to the fact that you're facing f...
...,...,...,...
26867,I am waiting for a rebate of {{Refund Amount}}...,REFUND,Thank you for sharing your situation regarding...
26868,how to see if there is anything wrong with my ...,REFUND,Ensuring the accuracy of your restitution is o...
26869,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,"Firstly, I genuinely understand the importance..."
26870,I don't know what to do to see my reimbursemen...,REFUND,I've understood you're unsure about how to che...


In [ ]:
df.to_csv('cleaned_data.csv')

In [7]:
df['category'].unique()


array(['ORDER', 'SHIPPING', 'CANCEL', 'INVOICE', 'PAYMENT', 'REFUND',
       'FEEDBACK', 'CONTACT', 'ACCOUNT', 'DELIVERY', 'SUBSCRIPTION'],
      dtype=object)

In [9]:
# Install required libraries
!pip install -q transformers datasets scikit-learn accelerate

In [10]:
import torch

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Additional GPU details
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

Using device: cuda
GPU Name: Tesla T4
GPU Count: 1


In [11]:
!nvidia-smi

Fri Apr 10 19:33:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"
num_labels = 11  # Based on your category column

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model and move to GPU
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)

print("✅ BERT model loaded successfully on GPU.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BERT model loaded successfully on GPU.


In [13]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# Step 1: Select only required columns
df_routing = df[['instruction', 'category']].dropna().copy()

# Rename columns for compatibility with Hugging Face
df_routing = df_routing.rename(columns={
    'instruction': 'text',
    'category': 'label'
})

print(f"Total samples: {len(df_routing)}")
print("Unique categories:", df_routing['label'].unique())

Total samples: 26872
Unique categories: ['ORDER' 'SHIPPING' 'CANCEL' 'INVOICE' 'PAYMENT' 'REFUND' 'FEEDBACK'
 'CONTACT' 'ACCOUNT' 'DELIVERY' 'SUBSCRIPTION']


In [14]:
df_routing

,text,label
0,question about cancelling order {{Order Number}},ORDER
1,i have a question about cancelling oorder {{Or...,ORDER
2,i need help cancelling puchase {{Order Number}},ORDER
3,I need to cancel purchase {{Order Number}},ORDER
4,"I cannot afford this order, cancel purchase {{...",ORDER
...,...,...
26867,I am waiting for a rebate of {{Refund Amount}}...,REFUND
26868,how to see if there is anything wrong with my ...,REFUND
26869,I'm waiting for a reimbjrsement of {{Currency ...,REFUND
26870,I don't know what to do to see my reimbursemen...,REFUND


In [15]:
# Initialize Label Encoder
label_encoder = LabelEncoder()

# Convert category labels to numerical IDs
df_routing['label'] = label_encoder.fit_transform(df_routing['label'])

# Create mappings
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("\nLabel to ID Mapping:")
print(label2id)


Label to ID Mapping:
{'ACCOUNT': 0, 'CANCEL': 1, 'CONTACT': 2, 'DELIVERY': 3, 'FEEDBACK': 4, 'INVOICE': 5, 'ORDER': 6, 'PAYMENT': 7, 'REFUND': 8, 'SHIPPING': 9, 'SUBSCRIPTION': 10}


In [16]:
train_df, val_df = train_test_split(
    df_routing,
    test_size=0.1,
    stratify=df_routing['label'],
    random_state=42
)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Training samples: 24184
Validation samples: 2688


In [17]:
# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Create a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 24184
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2688
    })
})


In [18]:
print("\nTraining Label Distribution:")
print(train_df['label'].value_counts())

print("\nValidation Label Distribution:")
print(val_df['label'].value_counts())


Training Label Distribution:
label
0     5387
6     3589
8     2693
5     1799
2     1799
7     1798
4     1797
3     1795
9     1773
10     899
1      855
Name: count, dtype: int64

Validation Label Distribution:
label
0     599
6     399
8     299
7     200
5     200
4     200
2     200
3     199
9     197
10    100
1      95
Name: count, dtype: int64


In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [20]:
model_name = "bert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [21]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True  # Ensures sequences do not exceed BERT's limit (512 tokens)
    )

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

In [23]:
from transformers import DataCollatorWithPadding

# Create a data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✅ Data collator created successfully.")

✅ Data collator created successfully.


In [24]:
from transformers import AutoModelForSequenceClassification
import torch

# Define the model name
model_name = "bert-base-uncased"

# Number of unique categories
num_labels = len(label2id)

# Load the BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ BERT model loaded on {device}.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BERT model loaded on cuda.


In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-routing-agent",   # Directory to save model checkpoints
    eval_strategy="epoch",         # Evaluate at the end of each epoch
    save_strategy="epoch",               # Save model at the end of each epoch
    learning_rate=2e-5,                  # Standard learning rate for BERT fine-tuning
    per_device_train_batch_size=16,      # Adjust based on GPU memory (e.g., 8 or 16 for T4)
    per_device_eval_batch_size=16,
    num_train_epochs=3,                  # Typically 3–5 epochs work well
    weight_decay=0.01,                   # Regularization to prevent overfitting
    logging_dir="./logs",                # Directory for training logs
    logging_steps=50,                    # Log training metrics every 50 steps
    load_best_model_at_end=True,         # Load the best model based on evaluation
    metric_for_best_model="f1",          # Metric used to select the best model
    greater_is_better=True,
    fp16=True                            # Enables mixed precision for faster GPU training
)

print("✅ Training arguments defined successfully.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training arguments defined successfully.


In [27]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    # Calculate metrics
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

print("✅ Evaluation metrics function defined.")

✅ Evaluation metrics function defined.


In [29]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer initialized successfully.")

✅ Trainer initialized successfully.


In [30]:
# Start training
trainer.train()

print("✅ Training completed successfully.")

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.015825,0.002212,0.999256,0.999260,0.999256,0.999255
2,0.001971,0.000577,1.000000,1.000000,1.000000,1.000000
3,0.000335,0.000339,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

✅ Training completed successfully.


In [31]:
from huggingface_hub import login

# Login to Hugging Face
login()  # Paste your Hugging Face token when prompted

In [32]:
trainer.push_to_hub("bert-routing-agent")
tokenizer.push_to_hub("bert-routing-agent")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g-agent/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...g-agent/training_args.bin:  11%|#         |   565B / 5.20kB            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/SandipanMondal06/bert-routing-agent/commit/21981152a0cb813525b12de4e3f5c321f3e23920', commit_message='Upload tokenizer', commit_description='', oid='21981152a0cb813525b12de4e3f5c321f3e23920', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SandipanMondal06/bert-routing-agent', endpoint='https://huggingface.co', repo_type='model', repo_id='SandipanMondal06/bert-routing-agent'), pr_revision=None, pr_num=None)

In [33]:
import torch

def predict_department(query):
    inputs = tokenizer(query, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    predicted_class_id = torch.argmax(probs, dim=1).item()
    confidence = probs[0][predicted_class_id].item()

    predicted_label = id2label[predicted_class_id]

    return predicted_label, confidence

# Example prediction
query = "I want to cancel my order."
label, confidence = predict_department(query)

print(f"Query: {query}")
print(f"Predicted Department: {label}")
print(f"Confidence: {confidence:.4f}")

Query: I want to cancel my order.
Predicted Department: ORDER
Confidence: 0.9995


In [34]:
query = "I have ordered 3 days ago but I did not get any tracking"
label, confidence = predict_department(query)

In [35]:
label

'ORDER'

In [36]:
confidence

0.591260552406311

In [37]:
query = "I have ordered 3 days ago but I did not get any tracking want my money back"
label, confidence = predict_department(query)

In [38]:
label

'REFUND'

In [39]:
confidence

0.9996639490127563